# MediaSearch — mediasearch.online

A focused web crawler and search engine for film industry news. This notebook documents the architecture, demonstrates each component, and provides a workspace for experiments.

## Architecture

```
seeds/film_news.txt     →  Crawler (BFS + robots.txt + polite delay)
                               ↓
                        data/pages/<hash>/
                            ├── page.html
                            └── metadata.json
                               ↓
                        Indexer (TF-IDF + title boost)
                               ↓
                        data/index/
                            ├── inverted_index.json
                            └── documents.json
                               ↓
                        Search (CLI or Flask web UI)
```

### Key Features
- **BFS crawl** with configurable depth and per-domain page limits
- **robots.txt** compliance and polite crawl delays
- **URL normalization** — strips tracking params, trailing slashes, fragments
- **SimHash content dedup** — detects near-duplicate pages across different URLs
- **Log-normalized TF-IDF** scoring with title boost
- **Flask web UI** at `https://mediasearch.online`

## Setup

In [ ]:
import json
import logging
import os
from collections import Counter

from crawler import Crawler
from indexer import Indexer
from search import search

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(message)s",
    datefmt="%H:%M:%S",
)

import config
print(f"Data dir:  {config.DATA_DIR}")
print(f"Pages dir: {config.PAGES_DIR}")
print(f"Index dir: {config.INDEX_DIR}")
print(f"Seeds:     {config.DEFAULT_SEED_FILE}")

## 1. Crawling

Run a small crawl (adjust `max_pages` and `max_depth` to control scope).

In [ ]:
crawler = Crawler(max_pages=5, max_depth=1)
pages_crawled = crawler.crawl()
print(f"\nVisited URLs: {len(crawler.visited)}")
print(f"Content fingerprints stored: {len(crawler.content_fingerprints)}")

## 2. Inspect Crawled Pages

Review what was fetched — titles, sizes, and crawl timestamps.

In [ ]:
pages = []
for d in os.listdir(config.PAGES_DIR):
    meta_path = os.path.join(config.PAGES_DIR, d, "metadata.json")
    if os.path.exists(meta_path):
        with open(meta_path) as f:
            pages.append(json.load(f))

print(f"{'Title':<55} {'Size':>8}  {'Depth':>5}  Crawled At")
print("-" * 100)
for p in sorted(pages, key=lambda x: x["crawled_at"]):
    title = (p["title"][:52] + "...") if len(p["title"]) > 55 else p["title"]
    print(f"{title:<55} {p['content_length']:>8,}  {p['depth']:>5}  {p['crawled_at']}")

## 3. Indexing

Build the TF-IDF inverted index from crawled pages.

In [ ]:
indexer = Indexer()
indexer.build()

# Show top 20 terms by number of documents they appear in
term_doc_counts = {term: len(entries) for term, entries in indexer.index.items()}
top_terms = Counter(term_doc_counts).most_common(20)
print(f"\nTop 20 most widespread terms (appear in N documents):")
for term, count in top_terms:
    print(f"  {term:<20} {count} docs")

## 4. Search

Query the index and view ranked results.

In [ ]:
queries = ["oscar awards", "box office", "independent film", "streaming"]

for q in queries:
    results = search(q)
    print(f'"{q}" — {len(results)} results')
    for r in results[:3]:
        print(f"  [{r['score']:6.2f}] {r['title'][:60]}")
    print()

## 5. Deduplication Demo

Test URL normalization and SimHash content fingerprinting.

In [ ]:
# URL normalization
print("=== URL Normalization ===\n")
test_urls = [
    "https://Example.COM/article/123/",
    "https://example.com/article/123?utm_source=twitter&page=2",
    "https://example.com/article/123?page=2#comments",
]
for url in test_urls:
    print(f"  {url}")
    print(f"  → {Crawler.normalize_url(url)}\n")

# SimHash near-duplicate detection
print("=== SimHash Content Fingerprinting ===\n")
texts = {
    "A": "New Marvel movie breaks opening weekend box office records",
    "B": "New Marvel film breaks opening weekend box office record",  # near-dupe
    "C": "Scientists discover high levels of microplastics in the ocean",  # different
}
hashes = {k: Crawler._simhash(v) for k, v in texts.items()}
for k, v in texts.items():
    print(f"  [{k}] {v}")
print()
for a in texts:
    for b in texts:
        if a < b:
            dist = Crawler._hamming_distance(hashes[a], hashes[b])
            label = "NEAR-DUPE" if dist <= 3 else "different"
            print(f"  {a}↔{b}: hamming distance = {dist}  ({label})")

## 6. Scratch Space

Use this area for experiments — try different queries, tweak scoring, test new seeds.

In [ ]:
# Try your own queries here
results = search("your query here")
for r in results:
    print(f"[{r['score']:6.2f}] {r['title']}")
    print(f"        {r['url']}\n")